# 04 · Detección de bajo rendimiento

**Residuo** = potencia real − potencia esperada **con la planta limpia** (`dias_desde_lluvia = 0`),
normalizado por la capacidad. Así, lo que queda debajo de cero es pérdida: falla o suciedad.

Comparamos dos familias de detectores:

| Familia | Señal diaria | Alertas |
|---|---|---|
| **Contra el propio modelo** | residuo relativo = (real − esperado) / esperado | umbral (mediana − 3σ robusta) y CUSUM |
| **Contra las vecinas** (método del notebook de referencia) | índice relativo = (real/esperado) / mediana de las vecinas del mismo día / "normal" de la planta | falla grave (< 0,5), CUSUM e Isolation Forest |

**Criterio de la hipótesis** (`docs/plan_modelado.md`): precisión y recall por día ≥ 0,8 para las
caídas abruptas de 10%, 25%, 50% y 100%, evaluando cada nivel por separado en los meses reservados
(mayo y junio de 2019). El detector se elige antes, con fallas inyectadas en jul-2018 a abr-2019.

**Argentina:** es una **comparación mensual entre parques** para marcar los que rinden por debajo de lo
esperado según la irradiación. **No es detección en tiempo real.**

In [ ]:
# Si corrés en Colab: clonar el repo (una vez por sesión), pararse en él e instalar dependencias
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/tpo-solar'):
        !git clone https://github.com/imelinc/tpo-solar.git /content/tpo-solar
    %cd /content/tpo-solar
    !pip install -q -r requirements.txt
sys.path.append('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
from src.config import DATA_RAW, DATA_PROCESSED, MODELS

In [ ]:
import warnings, zlib
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict
from src.modelos import ModeloPotencia, preparar, filas_entrenamiento, cv_por_mes, EXCLUIDAS
from src.deteccion import (residuo_diario, calibrar, ajustar_h_cusum, detectar, distancias_km, vecinas_por_distancia,
                           indice_relativo, calibrar_combinado, alertas_combinadas, diario_flota, semaforo_flota)
from src.fallas import inyectar_caida_abrupta, inyectar_soiling
from src.negocio import PRECIO_USD_MWH
from src.features import filtro_rendimiento_argentina

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200)
CORTE_HOLDOUT = "2019-05-01"
META = joblib.load(MODELS / "modelo_potencia.pkl")   # correr 03_modelos antes (el .pkl no está en git)
TIPO, ESC, LLUVIA = META["tipo"], META["escenario"], META["con_lluvia"]
print("Modelo elegido en 03:", TIPO, "| escenario:", ESC, "| con días desde lluvia:", LLUVIA)
A = pd.read_parquet(DATA_PROCESSED / "dataset_A_pvod_hourly.parquet")
d = preparar(A, ESC)
ok = filas_entrenamiento(d)
dev = d[ok & (d["timestamp_local_utc8"] < CORTE_HOLDOUT)].copy()
hold = d[ok & (d["timestamp_local_utc8"] >= CORTE_HOLDOUT)].copy()
REFS = sorted(dev["plant_id"].unique())   # plantas sanas de referencia (las del entrenamiento)

## 1. Lo esperado fuera de muestra

Para calibrar y evaluar sin trampa: en jul-2018 a abr-2019 se usan las predicciones de la validación cruzada por mes (cada mes lo predice un modelo que no lo vio). En mayo y junio se usa un modelo entrenado solo con jul-2018 a abr-2019. Para station09 (fuera del entrenamiento) se usa el modelo **planta típica**.

In [ ]:
kw = dict(con_lluvia=LLUVIA)
dev["esperado"] = cv_por_mes(dev, TIPO, ESC, limpia=True, **kw)
m_dev = ModeloPotencia(TIPO, ESC, **kw).fit(dev)
hold["esperado"] = m_dev.predict(hold, limpia=True)
# Planta típica: residuos fuera de muestra para calibrar el detector genérico
dev["esperado_gen"] = cv_por_mes(dev, TIPO, ESC, limpia=True, con_planta=False, **kw)
E_dev = residuo_diario(dev, dev["esperado"])
REF_E = E_dev.groupby("plant_id")["e_esp"].quantile(0.9).to_dict()
E_hold = residuo_diario(hold, hold["esperado"], REF_E)
E_gen = residuo_diario(dev, dev["esperado_gen"])
print(f"Días-planta dev: {len(E_dev)}, con decisión {E_dev['residuo_rel'].notna().mean():.0%} | "
      f"holdout: {len(E_hold)}, con decisión {E_hold['residuo_rel'].notna().mean():.0%}")
print("Sigma robusta del residuo relativo diario por planta (dev):")
print(E_dev.groupby("plant_id")["residuo_rel"].apply(lambda r: 1.4826 * (r - r.median()).abs().median()).round(3))

## 2. ¿Qué tan cerca están las plantas? (antes de compararlas con sus vecinas)

In [ ]:
meta_p = A.groupby("plant_id")[["plant_lat", "plant_lon"]].first()
DIST = distancias_km(meta_p["plant_lat"], meta_p["plant_lon"])
print(DIST.round(0).astype(int))
# ¿Se parecen los errores diarios del modelo entre plantas cercanas? Correlación del índice real/esperado
ind = (E_dev.assign(i=E_dev["e_real"] / E_dev["e_esp"]).where(E_dev["residuo_rel"].notna())
       .pivot_table(index="date_local", columns="plant_id", values="i"))
pares = [(a, b, DIST.loc[a, b], ind[a].corr(ind[b])) for i, a in enumerate(REFS) for b in REFS[i + 1:]]
pares = pd.DataFrame(pares, columns=["a", "b", "km", "corr_indice"])
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(pares["km"], pares["corr_indice"])
for _, r in pares.iterrows():
    ax.annotate(f"{r.a[-2:]}-{r.b[-2:]}", (r.km, r.corr_indice), fontsize=7)
ax.axvline(100, color="red", ls="--", label="radio elegido: 100 km")
ax.set(xlabel="distancia (km)", ylabel="correlación del índice diario real/esperado",
       title="Plantas cercanas comparten el error del modelo (nubes que el sensor/modelo no ven igual)"); ax.legend()
plt.show()
print(f"Correlación media: < 100 km {pares.loc[pares.km < 100, 'corr_indice'].mean():.2f} | "
      f">= 100 km {pares.loc[pares.km >= 100, 'corr_indice'].mean():.2f}")
VECINAS = vecinas_por_distancia(DIST, REFS, radio_km=100)
print("Vecinas (referencias a < 100 km):", VECINAS)

**Conclusión:** Las plantas forman dos grupos: el **oeste** (00, 02, 03 y 05, a 9-75 km entre sí) y el **sur** (06, 07 y 08, a 24-86 km). station01 y station04 no tienen ninguna referencia a menos de 100 km. Aun así, el índice diario de plantas cercanas casi no se correlaciona: 0,12 a menos de 100 km contra 0,07 más lejos. Con el sensor de cada planta, el modelo ya absorbe el clima local, y lo que queda es ruido propio de cada planta. Por eso dividir por las vecinas **suma** ruido en vez de cancelarlo. El método del notebook de referencia tiene más sentido con clima satelital, donde el error es común a toda la zona.

## 3. Calibración (jul-2018 a abr-2019, días sanos) y detector guardado

El umbral es mediana − 3σ robusta. Para el CUSUM, la regla de bolsillo (k = 0,5σ, h = 4σ) supone ruido independiente, pero el residuo diario está autocorrelado (un día malo suele venir con otros) y su nivel cambia de un mes a otro. Por eso **h se calibra** para que en estos días sanos haya como mucho 1 falsa alarma por planta y por mes (`ajustar_h_cusum`). La calibración usa solo jul-2018 a abr-2019.

In [ ]:
PROPIO = {p: calibrar(g["residuo_rel"]) | {"ref_e_esp": REF_E[p]} for p, g in E_dev.groupby("plant_id")}
PROPIO_GEN = calibrar(E_gen["residuo_rel"]) | {"ref_e_esp": float(E_gen["e_esp"].quantile(0.9))}
PROPIO, F_PROPIO = ajustar_h_cusum({p: g["residuo_rel"] for p, g in E_dev.groupby("plant_id")}, PROPIO)
PROPIO_GEN["h"] = F_PROPIO * PROPIO_GEN["sigma"]
# "Normal" de cada planta = mediana de su cociente contra las vecinas en días sanos
NORMAL = indice_relativo(E_dev, VECINAS, normal={p: 1.0 for p in d["plant_id"].unique()}).groupby("plant_id")["indice_rel"].median().to_dict()
I_dev = indice_relativo(E_dev, VECINAS, NORMAL)
COMB = calibrar_combinado(I_dev)
COMB["cusum"], F_VEC = ajustar_h_cusum({p: g["indice_rel"] - 1 for p, g in I_dev.groupby("plant_id")}, COMB["cusum"])
COMB["cusum_generico"]["h"] = F_VEC * COMB["cusum_generico"]["sigma"]
print(f"h del CUSUM calibrado: propio = {F_PROPIO}σ, vecinas = {F_VEC}σ (la regla de bolsillo era 4σ)")
print("Umbral propio (residuo relativo) por planta:", {p: round(v["umbral"], 3) for p, v in PROPIO.items()})
print("Normal frente a vecinas:", {p: round(v, 3) for p, v in NORMAL.items()})
DETECTOR = {"ref_e_esp": REF_E | {p: PROPIO_GEN["ref_e_esp"] for p in d["plant_id"].unique() if p not in REF_E},
            "propio": PROPIO, "propio_generico": PROPIO_GEN, "vecinas": VECINAS,
            "normal": NORMAL | {p: 1.0 for p in d["plant_id"].unique() if p not in NORMAL},
            "combinado": COMB, "radio_km": 100, "metodo": None}

## 4. Falsas alarmas en días sanos

In [ ]:
def todas_las_alertas(E):
    E = detectar(E, {p: PROPIO.get(p, PROPIO_GEN) for p in E["plant_id"].unique()})
    X = alertas_combinadas(indice_relativo(E, VECINAS, DETECTOR["normal"]), COMB)
    X["propio_umbral_o_cusum"] = X["alerta_umbral"] | X["alerta_cusum"]
    X["combinado_vecinas"] = X["alerta"]
    return X

METODOS = {"alerta_umbral": "Umbral (propio)", "alerta_cusum": "CUSUM (propio)",
           "propio_umbral_o_cusum": "Umbral o CUSUM (propio)", "alerta_grave": "Falla grave (vecinas < 0,5)",
           "alerta_cusum_rel": "CUSUM (vecinas)", "alerta_iso": "Isolation Forest (vecinas)",
           "combinado_vecinas": "Combinado vecinas (grave o CUSUM o IF)"}
X_dev, X_hold = todas_las_alertas(E_dev), todas_las_alertas(E_hold)
fa = pd.DataFrame({
    "dev (calibración)": {METODOS[m]: X_dev[m].sum() / (len(X_dev) / 30) for m in METODOS},
    "mayo-junio (fuera de muestra)": {METODOS[m]: X_hold[m].sum() / (len(X_hold) / 30) for m in METODOS}})
print("Falsas alarmas por planta y por mes (días sin falla inyectada):"); print(fa.round(2))
X_dev["mes"] = pd.to_datetime(X_dev["date_local"]).dt.month
fig, ax = plt.subplots(figsize=(11, 3.8))
sns.heatmap(X_dev.pivot_table(index="plant_id", columns="mes", values="combinado_vecinas", aggfunc="sum"),
            annot=True, fmt=".0f", cmap="Oranges", ax=ax)
ax.set(title="Días con alerta (combinado vecinas) por planta y mes, sin fallas inyectadas (jul-2018 a abr-2019)", xlabel="mes", ylabel="")
plt.show()

**Conclusión:** En mayo y junio, sin fallas inyectadas, el umbral da **0,17 falsas alarmas por planta y por mes**. El CUSUM con la regla de bolsillo (h = 4σ) daba entre 6 y 7 por mes. Para bajarlo a 1 por mes en la calibración hubo que subir h a 42σ (propio) y 40σ (vecinas). La causa es que el residuo diario está autocorrelado (lag 1 de 0,4 a 0,46) y su nivel cambia hasta ±7% de un mes a otro, así que el CUSUM confunde esa deriva del modelo con una pérdida sostenida. Las alertas sin falla se concentran en **station06 de enero a marzo** (59 días con alerta combinada), la limitación invernal que ya se anticipó en `03`.

## 5. Fallas sintéticas

Para cada planta de prueba, nivel y duración se inyecta una falla con `src/fallas.py` sobre la potencia real horaria, en una fecha al azar (5 semillas). Un día cuenta como **con falla** si la pérdida media inyectada es ≥ 3%. La precisión cuenta como falsas alarmas a las alertas en días sin falla de la misma ventana, incluidas las que siguen encendidas después de que termina.

In [ ]:
def simular(H, E_base, tipo, valor, dias, semillas=5, rng0=0):
    """Inyecta la falla en cada planta (una por vez) y devuelve una fila por día de esa planta."""
    filas, resumen = [], []
    for p in REFS:
        h = H[H["plant_id"] == p]
        fechas = sorted(h["date_local"].unique())
        if len(fechas) < dias + 5:
            continue
        for s in range(semillas):
            rng = np.random.default_rng(zlib.crc32(f"{p}-{tipo}-{valor}-{dias}-{s}-{rng0}".encode()))
            ini = pd.Timestamp(fechas[rng.integers(2, len(fechas) - dias)])
            serie = h.set_index("timestamp_local_utc8")["pvod_power_norm"]
            f = inyectar_caida_abrupta if tipo == "abrupta" else inyectar_soiling
            pf, perdida = f(serie, ini, dias, valor)
            dia = h["date_local"].to_numpy()
            nuevo = pd.Series(pf.to_numpy(), index=h.index).groupby(dia).sum()
            perd_dia = pd.Series(perdida.to_numpy(), index=h.index).groupby(dia).mean()
            E = E_base.copy()
            m = E["plant_id"] == p
            E.loc[m, "e_real"] = E.loc[m, "date_local"].map(nuevo).to_numpy()
            E.loc[m, "residuo"] = E.loc[m, "e_real"] - E.loc[m, "e_esp"]
            E.loc[m, "residuo_rel"] = (E.loc[m, "residuo"] / E.loc[m, "e_esp"]).where(E.loc[m, "residuo_rel"].notna())
            X = todas_las_alertas(E)
            X = X[X["plant_id"] == p].copy()
            X["perdida"] = X["date_local"].map(perd_dia).fillna(0).to_numpy()
            X["con_falla"] = X["perdida"] >= 0.03
            X["decide"] = X["residuo_rel"].notna()
            X["mwh_inyectados"] = (pd.Series(((serie - pf) * h["plant_capacity_mw"].to_numpy()).to_numpy(), index=h.index)
                                   .groupby(dia).sum().reindex(X["date_local"]).fillna(0).to_numpy())
            X["sim"] = f"{p}-{s}"; X["tipo"], X["valor"], X["dias"], X["inicio"] = tipo, valor, dias, ini
            filas.append(X)
    return pd.concat(filas, ignore_index=True)


def evaluar(R, metodo):
    pos = R["con_falla"] & R["decide"]
    tp = (R[metodo] & pos).sum(); fp = (R[metodo] & ~R["con_falla"] & R["decide"]).sum()
    prec = tp / (tp + fp) if tp + fp else np.nan
    rec = tp / pos.sum() if pos.sum() else np.nan
    demoras, perd_det = [], []
    for _, g in R.groupby("sim"):
        g = g.sort_values("date_local")
        en = g[(pd.to_datetime(g["date_local"]) >= g["inicio"].iloc[0]) & g[metodo] & g["con_falla"]]
        if len(en):
            demoras.append((pd.Timestamp(en["date_local"].iloc[0]) - g["inicio"].iloc[0]).days)
            perd_det.append(en["perdida"].iloc[0])
    sims = R["sim"].nunique()
    return {"precision": prec, "recall": rec, "F1": 2 * prec * rec / (prec + rec) if prec + rec else np.nan,
            "detectadas_%": 100 * len(demoras) / sims, "demora_mediana_dias": np.median(demoras) if demoras else np.nan,
            "perdida_al_detectar_%": 100 * np.median(perd_det) if perd_det else np.nan,
            "dias_sin_decision_%": 100 * (R["con_falla"] & ~R["decide"]).sum() / max(R["con_falla"].sum(), 1)}

NIVELES, DURACIONES, TASAS = [0.05, 0.10, 0.25, 0.50, 1.0], [1, 3, 7], [0.001, 0.002, 0.005]

### 5.1 Elección del detector (fallas inyectadas en jul-2018 a abr-2019)

In [ ]:
R_dev = pd.concat([simular(dev, E_dev, "abrupta", f, dd, semillas=3, rng0=1) for f in NIVELES[1:] for dd in DURACIONES])
sel = pd.DataFrame({METODOS[m]: {f"{int(f*100)}%": evaluar(R_dev[R_dev["valor"] == f], m)["F1"] for f in NIVELES[1:]}
                    for m in METODOS}).T
sel["F1_minimo"] = sel.min(axis=1)
print("F1 por nivel de caída (dev):"); print(sel.round(2).sort_values("F1_minimo", ascending=False))
METODO = {v: k for k, v in METODOS.items()}[sel["F1_minimo"].idxmax()]
DETECTOR["metodo"] = METODO
print("\nDetector elegido (mejor F1 en el peor nivel):", METODOS[METODO])
joblib.dump(DETECTOR, MODELS / "detector.pkl")
print("Guardado:", MODELS / "detector.pkl")

**Conclusión:** Con las fallas inyectadas en jul-2018 a abr-2019, el **umbral sobre el residuo propio** es el que tiene mejor F1 en el peor nivel: 0,12 en 10% y entre 0,53 y 0,66 de 25% a 100%. Las alertas contra vecinas solo sirven para caídas grandes: la falla grave llega a un F1 de 0,86 en 100%, pero da 0,04 en 10% y 25%. El CUSUM calibrado casi no detecta. Se elige **Umbral (propio)** y se guarda en `models/detector.pkl`.

### 5.2 ⭐ Evaluación final en mayo y junio de 2019 (una sola vez)

In [ ]:
R = pd.concat([simular(hold, E_hold, "abrupta", f, dd) for f in NIVELES for dd in DURACIONES])
tabla = []
for m in [METODO, "propio_umbral_o_cusum", "combinado_vecinas", "alerta_grave", "alerta_cusum_rel", "alerta_iso", "alerta_umbral", "alerta_cusum"]:
    for f in NIVELES:
        tabla.append({"detector": METODOS[m], "caida": f"{int(f*100)}%", **evaluar(R[R["valor"] == f], m)})
tabla = pd.DataFrame(tabla).drop_duplicates(["detector", "caida"])
print(tabla.round(2).to_string(index=False))
fin = tabla[tabla["detector"] == METODOS[METODO]].set_index("caida")
cumple = (fin.loc[["10%", "25%", "50%", "100%"], ["precision", "recall"]] >= 0.8).all().all()
print(f"\nCriterio de la hipótesis (precisión y recall >= 0,8 en 10/25/50/100%) con {METODOS[METODO]}: "
      f"{'CUMPLE' if cumple else 'NO CUMPLE'}")
print(fin.loc[["10%", "25%", "50%", "100%"], ["precision", "recall"]].round(2))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
sel_m = [METODOS[METODO], "Umbral o CUSUM (propio)", "Combinado vecinas (grave o CUSUM o IF)"]
for ax, met in zip(axs, ["precision", "recall"]):
    sns.lineplot(data=tabla[tabla["detector"].isin(sel_m)], x="caida", y=met, hue="detector", marker="o", ax=ax)
    ax.axhline(0.8, color="red", ls="--", lw=1)
    ax.set(title=met.capitalize() + " por día, mayo-junio 2019", xlabel="caída inyectada", ylim=(0, 1.02), ylabel="")
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

**Conclusión:** En mayo y junio, con el umbral: caída de 100%, precisión 0,93 y recall 1,00; 50%, 0,94 y 0,95; 25%, 0,92 y 0,73; 10%, 0,68 y 0,12. **La condición 2 no se cumple** para 10% y 25%. Ningún otro detector cumple: combinar el umbral con el CUSUM o con las vecinas empeora la precisión sin mejorar el recall en 10% y 25%.

**La hipótesis se sostiene parcialmente.** Se confirma para fallas grandes y casos crónicos, y se refuta para pérdidas chicas:
- **Pérdidas ≥ 50%:** el detector las encuentra el mismo día (precisión ≥ 0,93, recall ≥ 0,95).
- **Casos crónicos:** marca station09 (desde su segundo día con datos) y ULL4FV (11 de 12 meses de 2025).
- **Caídas de 25%:** quedan en el límite (recall 0,73: se escapa 1 de cada 4 días).
- **Caídas de 10% día a día (y suciedad gradual):** no las detecta (recall 0,12), porque el error diario del modelo (5% a 16% según la planta) es del mismo orden que la pérdida.


### 5.3 Pérdida gradual por suciedad (soiling) y energía perdida

In [ ]:
Rs = pd.concat([simular(hold, E_hold, "soiling", r, 30) for r in TASAS])
ts = pd.DataFrame([{"detector": METODOS[m], "tasa_%_dia": r * 100, **evaluar(Rs[Rs["valor"] == r], m)}
                   for m in [METODO, "alerta_cusum", "alerta_cusum_rel", "combinado_vecinas"] for r in TASAS]).drop_duplicates(["detector", "tasa_%_dia"])
print(ts[["detector", "tasa_%_dia", "detectadas_%", "demora_mediana_dias", "perdida_al_detectar_%", "precision", "recall"]].round(2).to_string(index=False))
# Energía: inyectada vs. la que estima el detector en los días con alerta
todo = pd.concat([R, Rs])
todo["mwh_estimados"] = (-todo["residuo"] * todo["cap_mw"]).clip(lower=0).where(todo[METODO], 0)
mwh = todo.groupby(["tipo", "valor"]).agg(mwh_inyectados=("mwh_inyectados", "sum"), mwh_estimados_en_alerta=("mwh_estimados", "sum"))
mwh["capturado_%"] = 100 * mwh["mwh_estimados_en_alerta"] / mwh["mwh_inyectados"]
print("\nMWh perdidos: inyectados vs. estimados por el detector en los días con alerta"); print(mwh.round(1))
# Para 05_negocio: demoras y detección por tipo de falla con el detector elegido
pd.concat([tabla[tabla["detector"] == METODOS[METODO]].assign(tipo="abrupta"),
           ts[ts["detector"] == METODOS[METODO]].rename(columns={"tasa_%_dia": "caida"}).assign(tipo="soiling")]
          ).to_csv(MODELS / "evaluacion_detector.csv", index=False)

**Conclusión:** La suciedad gradual casi no se detecta. A 0,5%/día, el umbral la encuentra en el 42,5% de los casos, con una demora mediana de 19 días, cuando ya se perdía cerca de 9,7%; a 0,1%/día no la encuentra nunca. La energía perdida se estima bien en las caídas grandes (99% de lo inyectado en 50% y 83% en 25%), pero en las de 10% se captura solo el 38%, y en suciedad entre 13% y 26%. **Para la suciedad, la herramienta útil no es la alerta diaria sino la limpieza programada** (`05`).

## 6. Casos reales

### 6.1 ⭐ station09: bajo rendimiento crónico y recuperación en escalones

In [ ]:
E9 = diario_flota("2018-09-26", "2019-06-13")   # usa models/ (modelo final y detector guardado)
s9 = E9[E9["plant_id"] == "station09"].copy()
s9["fecha"] = pd.to_datetime(s9["date_local"])
antes = s9[s9["fecha"] < "2019-05-19"]
recup = s9[s9["fecha"].between("2019-05-19", "2019-06-03")]   # después el sensor está caído
print(f"station09: índice relativo mediano antes del 19/5: {antes['indice_rel'].median():.2f} | "
      f"19/5 al 3/6 (recuperación): {recup['indice_rel'].median():.2f}")
for col in [METODO, "alerta"]:
    nom = METODOS.get(col, "Combinado vecinas")
    print(f"Días con decisión y alerta ({nom}): antes del 19/5 {antes.loc[antes['indice_rel'].notna(), col].mean():.0%} | "
          f"19/5 al 3/6 {recup.loc[recup['indice_rel'].notna(), col].mean():.0%}")
print(f"Primera alerta: {s9.loc[s9[METODO], 'date_local'].min()} (primer día con datos: {s9['date_local'].min()})")
print(f"Días con el sensor de irradiancia caído (sin decisión): {s9['falla_sensor'].sum()}")
print(f"Energía perdida estimada vs. planta típica: {s9['perdida_mwh'].sum():,.0f} MWh "
      f"(USD {s9['perdida_mwh'].sum() * PRECIO_USD_MWH:,.0f} a USD {PRECIO_USD_MWH}/MWh)")
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(s9["fecha"], s9["indice_rel"], color="grey", lw=1, label="índice relativo a vecinas")
for col, colr, lbl in [("alerta_grave", "red", "falla grave"), ("alerta_cusum_rel", "orange", "pérdida sostenida"), ("alerta_iso", "purple", "día anómalo")]:
    a = s9[s9[col]]; ax.scatter(a["fecha"], a["indice_rel"], color=colr, s=14, label=lbl, zorder=3)
ax.axhline(1, color="k", lw=0.8); ax.axhline(0.5, color="red", ls="--", lw=0.8)
ax.axvspan(pd.Timestamp("2019-06-04"), s9["fecha"].max(), color="grey", alpha=0.15, label="sensor caído (sin decisión)")
ax.set(title="station09 frente a sus vecinas: rinde la mitad hasta mayo y se recupera en escalones", ylabel="1 = como una planta típica")
ax.legend(fontsize=8, ncol=5); plt.show()

**Conclusión:** station09 se marca desde el segundo día con datos (27/9/2018). Antes del 19/5/2019, su índice relativo mediano es 0,41 (rinde el 41% de una planta típica) y el umbral alerta el 53% de los días con decisión; las alertas combinadas, el 100%. En la recuperación (19/5 al 3/6) el índice sube a 0,59 y el umbral deja de alertar, pero las combinadas siguen, porque todavía está muy por debajo de lo normal. Desde el 4/6 el sensor está caído y el sistema se abstiene (14 días sin decisión). Pérdida estimada contra una planta típica: **10.153 MWh, unos USD 435 mil a USD 42,84/MWh**. **La condición 3 se cumple para station09.**

### 6.2 Apagones con sol que ya estaban en los datos

In [ ]:
E_all = diario_flota("2018-06-30", "2019-06-13")
apag = d[d["apagon"] & d["plant_id"].isin(REFS)].groupby(["plant_id", "date_local"]).size().rename("horas_apagon").reset_index()
x = apag.merge(E_all, on=["plant_id", "date_local"], how="left")
print(f"Días-planta con horas de apagón: {len(x)} | con decisión: {x['indice_rel'].notna().sum()} | "
      f"con alerta del detector elegido: {x[METODO].sum()} | con cualquier alerta: {(x['alerta'] | x['alerta_umbral'] | x['alerta_cusum']).sum()}")
print(x[["plant_id", "date_local", "horas_apagon", "indice_rel", "tipo_alerta", METODO]].round(2).to_string(index=False))

**Conclusión:** Hay 9 días con horas de apagón con sol, que el modelo nunca vio en el entrenamiento, y el umbral alerta en 8. El único sin alerta es station05 el 1/5/2019, con una sola hora de apagón (índice de 0,87). Ojo: para el resto de esos días el modelo es dentro de muestra, así que este chequeo es optimista.

### 6.3 Semáforo de la flota (función para el dashboard)

In [ ]:
print(semaforo_flota("2019-05-10").round(2).to_string(index=False))

**Conclusión:** Al 10/5/2019, **station09 está en rojo** (falla grave: 0,44 de una planta típica, cerca de USD 2.300 por día), station07 en amarillo (0,93 frente a sus vecinas) y el resto en verde. station04 sale en **gris, "capacidad no confiable"**: su pico real es de 24,8 MW con 20 MW nominales (más de 1,1 veces), así que su 1,45 contra la planta típica no significa nada hasta corregir la ficha. En las plantas verdes, la pérdida en USD (por ejemplo, USD 117 por día en station06) se mide contra la planta limpia: es suciedad o estación, no una falla.

## 7. Argentina: comparación mensual entre parques (no es detección en tiempo real)

Lo esperado es el factor de capacidad de un **parque típico** con esa irradiación, temperatura, lluvia y mes
(Random Forest, sin el parque como variable). El índice de cada parque (real / esperado) se divide por la
mediana de los parques de **su región en el mismo mes**, y por su "normal" en el período de
entrenamiento. Se entrena hasta 2025-09 y se evalúa de 2025-10 a 2026-09.

In [ ]:
B = pd.read_parquet(DATA_PROCESSED / "dataset_B_argentina_mensual.parquet")
B = B[filtro_rendimiento_argentina(B)].copy()
B["fecha"] = pd.to_datetime(B["fecha"])
B["mes_sin"], B["mes_cos"] = np.sin(2 * np.pi * B["mes"] / 12), np.cos(2 * np.pi * B["mes"] / 12)
FB = ["om_ghi_kwh_m2_mes", "om_temp_max_media_c", "om_nubosidad_media_pct", "om_precip_mm_mes", "om_dias_lluvia_5mm", "mes_sin", "mes_cos", "lat"]
FB = [f for f in FB if f in B]
B = B.dropna(subset=FB + ["factor_capacidad"])
tr, te = B[B["fecha"] <= "2025-09-01"], B[B["fecha"] > "2025-09-01"]
rf_b = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(tr[FB], tr["factor_capacidad"])
B["fc_esp"] = np.nan
B.loc[te.index, "fc_esp"] = rf_b.predict(te[FB])
# En entrenamiento, predicciones fuera de muestra dejando parques enteros afuera (como un parque nuevo)
B.loc[tr.index, "fc_esp"] = cross_val_predict(
    RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1), tr[FB], tr["factor_capacidad"],
    groups=tr["central"], cv=GroupKFold(5))
mae_te = (te["factor_capacidad"] - B.loc[te.index, "fc_esp"]).abs().mean()
print(f"Features: {FB}")
print(f"MAE del factor de capacidad, parque típico: entrenamiento (CV por parque) {(tr['factor_capacidad'] - B.loc[tr.index, 'fc_esp']).abs().mean():.3f} | "
      f"2025-10 a 2026-09 {mae_te:.3f} (FC medio {te['factor_capacidad'].mean():.3f})")
B["indice"] = B["factor_capacidad"] / B["fc_esp"]
B["indice_reg"] = B["indice"] / B.groupby(["region", "fecha"])["indice"].transform("median")
normal_b = B[B["fecha"] <= "2025-09-01"].groupby("central")["indice_reg"].median()
B["indice_rel"] = B["indice_reg"] / B["central"].map(normal_b)
B = B.sort_values(["central", "fecha"])
from src.deteccion import alarmas_cusum
trb = B[B["fecha"] <= "2025-09-01"]
cal_b = {p: calibrar(g["indice_rel"] - 1) for p, g in trb.groupby("central") if len(g) >= 12}
# h calibrado: como mucho 1 mes con alerta por parque y por año en el período de entrenamiento
cal_b, F_B = ajustar_h_cusum({p: g["indice_rel"] - 1 for p, g in trb.groupby("central") if p in cal_b}, cal_b, tasa_max=1 / 12)
gen_b = calibrar(trb["indice_rel"] - 1); gen_b["h"] = F_B * gen_b["sigma"]
print(f"h del CUSUM mensual calibrado: {F_B}σ")
B["alerta_grave"] = B["indice_rel"] < 0.5
B["alerta_cusum"] = False
for p, idx in B.groupby("central").groups.items():
    B.loc[idx, "alerta_cusum"] = alarmas_cusum(B.loc[idx, "indice_rel"] - 1, cal_b.get(p, gen_b)).to_numpy()
B["alerta"] = B["alerta_grave"] | B["alerta_cusum"]
print(f"Meses-parque con alerta: {B['alerta'].sum()} de {len(B)} ({B['alerta'].mean():.1%}), en {B.loc[B['alerta'], 'central'].nunique()} parques")

In [ ]:
# Bajo su percentil histórico: índice relativo menor que el p10 del propio parque en el entrenamiento
p10_b = trb.groupby("central")["indice_rel"].quantile(0.10)
B["bajo_p10"] = B["indice_rel"] < B["central"].map(p10_b)
COLS_APP = ["fecha", "central", "central_descripcion", "provincia", "region", "lat", "lon", "potencia_mw", "energia_gwh",
            "factor_capacidad", "fc_esp", "indice_rel", "alerta_grave", "alerta_cusum", "alerta", "bajo_p10"]
B[COLS_APP].to_parquet(DATA_PROCESSED / "app_argentina.parquet", index=False)   # lo lee el dashboard
print(f"Meses-parque bajo su p10 histórico: {B['bajo_p10'].sum()} ({B['bajo_p10'].mean():.1%})")
u = B[B["central"] == "ULL4FV"]
a25 = u[u["fecha"].dt.year == 2025]
print(f"ULL4FV en 2025: meses con alerta {a25['alerta'].sum()} de {len(a25)} | índice relativo mediano {a25['indice_rel'].median():.2f} "
      f"(resto de la historia: {u.loc[u['fecha'].dt.year != 2025, 'indice_rel'].median():.2f})")
print(f"Primer mes con alerta en 2025: {a25.loc[a25['alerta'], 'fecha'].min():%Y-%m}")
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(u["fecha"], u["factor_capacidad"], ".-", label="FC real")
ax.plot(u["fecha"], u["fc_esp"], ".-", label="FC esperado (parque típico)")
ax.scatter(u.loc[u["alerta"], "fecha"], u.loc[u["alerta"], "factor_capacidad"], color="red", zorder=3, label="alerta")
ax.set(title="ULL4FV (San Juan): la comparación mensual marca 2025", ylabel="factor de capacidad"); ax.legend(); plt.show()
ult = B[B["fecha"] > "2025-09-01"].groupby(["central", "region"]).agg(
    indice_rel=("indice_rel", "median"), meses=("fecha", "size"), meses_alerta=("alerta", "sum"),
    energia_gwh=("energia_gwh", "sum")).query("meses >= 6").sort_values("indice_rel")
print("\nParques que más rinden por debajo de su región (2025-10 a 2026-09, mínimo 6 meses):")
print(ult.head(10).round(2))

**Conclusión:** Es una comparación mensual, no detección en tiempo real. El modelo de parque típico predice el factor de capacidad con un MAE de 0,040 (validación cruzada por parque) y de 0,043 en 2025-10 a 2026-09, con un FC medio de 0,252. Con h calibrado (20σ), el 10% de los meses-parque tienen alerta, repartidos en 42 parques. **ULL4FV tiene alerta en 11 de los 12 meses de 2025**, desde febrero, con un índice relativo de 0,15 contra 1,01 en el resto de su historia. **La condición 3 se cumple para ULL4FV.** En el último año, los parques más por debajo de su región son CHEPFV (0,74), VMR2FV (0,76) y TAMBFV (0,84): son candidatos para revisar con el área técnica (puede ser una falla o una potencia mal cargada).

## 8. Tablas para el dashboard

La app no carga los `.pkl` (no están en git): lee lo que se guarda acá en `data/processed/` (`app_diario.parquet`, `app_horario.parquet` y, en la sección 7, `app_argentina.parquet`).

In [ ]:
from src.deteccion import guardar_precalculado, PRECALC_DIARIO, PRECALC_HORARIO
guardar_precalculado()
for f in [PRECALC_DIARIO, PRECALC_HORARIO, DATA_PROCESSED / "app_argentina.parquet"]:
    print(f"{f.name}: {f.stat().st_size / 1e6:.1f} MB")

## Resumen

**Resultado de la hipótesis** (umbrales fijados antes de los meses reservados):

| Condición | Resultado | ¿Cumple? |
|---|---|---|
| 1. Error de energía diaria < 8% (sensor de planta, evaluación ciega) | 6,04% (ciego) | ✅ |
| 2. Precisión y recall ≥ 0,8 para caídas de 10%, 25%, 50% y 100% | ≥ 50% sí; 25% en el límite (recall 0,73); 10% no (recall 0,12) | ❌ |
| 3. Marca los casos reales (station09 y ULL4FV) | station09 desde el día 2; ULL4FV, 11 de 12 meses de 2025 | ✅ |

**La hipótesis se sostiene parcialmente.** Se confirma para fallas grandes y casos crónicos, y se refuta para pérdidas chicas:
- **Pérdidas ≥ 50%:** el detector las encuentra el mismo día (precisión ≥ 0,93, recall ≥ 0,95).
- **Casos crónicos:** marca station09 (desde su segundo día con datos) y ULL4FV (11 de 12 meses de 2025).
- **Caídas de 25%:** quedan en el límite (recall 0,73: se escapa 1 de cada 4 días).
- **Caídas de 10% día a día (y suciedad gradual):** no las detecta (recall 0,12), porque el error diario del modelo (5% a 16% según la planta) es del mismo orden que la pérdida.

> **Número principal: 6,04%** de error de energía diaria (evaluación ciega). *Nota:* el modelo final (híbrido con lluvia) da 5,04% en los mismos meses, pero **no es una evaluación ciega** (se agregó después de la primera mirada), así que no se usa como resultado.

**Gráficos para la defensa:** (1) error de energía diaria por modelo (`03`), (2) precisión y recall por tamaño de caída (5.2) y (3) station09 frente a sus vecinas (6.1).